In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
import sys
if sys.platform == 'win32':
    import asyncio
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
import gc
gc.collect()

In [ ]:
# Load JSON results from both models
with open("results_old.json", "r") as f:
    old = json.load(f)

with open("results_new.json", "r") as f:
    new = json.load(f)

print("Old model results:", list(old.keys()))
print("New model results:", list(new.keys()))


In [ ]:
# Convert to DataFrame for easier analysis
df_old = pd.DataFrame(old).T
df_new = pd.DataFrame(new).T

# Remove raw data since mesonet only starts from five minute aggregation
keep = ["five", "quarter", "hourly", "daily"]

df_old = df_old.loc[keep]
df_new = df_new.loc[keep]

print("\nNYISO-only model metrics:")
print(df_old)
print("\nNYISO + Mesonet model metrics:")
print(df_new)


In [ ]:
# Calculate improvement percentages
improvement = pd.DataFrame(index=keep)

for metric in ["MAPE", "MAE", "RMSE"]:
    # For these metrics, lower is better
    improvement[f"{metric}_improvement_%"] = ((df_old[metric] - df_new[metric]) / df_old[metric] * 100)

# For R², higher is better
improvement["R2_improvement_%"] = ((df_new["R2"] - df_old["R2"]) / df_old["R2"] * 100)

print("\nImprovement from adding Mesonet weather data:")
print(improvement.round(2))


## Comparison Plots
Visualize performance metrics across different time aggregations

In [ ]:
metrics = ["MAPE", "MAE", "RMSE", "R2"]
titles = {
    "MAPE": "MAPE (%)",
    "MAE":  "Mean Absolute Error (MAE)",
    "RMSE": "Root Mean Squared Error (RMSE)",
    "R2":   "R² Score"
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

x = np.arange(len(keep))  # positions
bar_width = 0.35

for i, metric in enumerate(metrics):
    ax = axes[i]

    old_vals = df_old[metric].values
    new_vals = df_new[metric].values

    bars1 = ax.bar(x - bar_width/2, old_vals, width=bar_width, 
                   label="NYISO Only Model", alpha=0.7, color='steelblue')
    bars2 = ax.bar(x + bar_width/2, new_vals, width=bar_width, 
                   label="NYISO + Mesonet Model", alpha=0.7, color='coral')

    ax.set_title(titles[metric], fontsize=14, weight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(keep, fontsize=12)
    ax.set_xlabel("Time Aggregation", fontsize=11)
    ax.set_ylabel(titles[metric], fontsize=11)
    ax.grid(axis="y", alpha=0.3)

    # Improve spacing for tight metrics
    if metric == "R2":
        ax.set_ylim(0, 1.05)

    ax.legend(loc='best')

plt.suptitle("Model Performance Comparison: NYISO vs NYISO+Mesonet", 
             fontsize=16, weight='bold', y=1.00)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=300, bbox_inches='tight')
plt.show()


## Summary Statistics

In [ ]:
# Calculate average improvement across all aggregations
avg_improvement = improvement.mean()
print("Average improvement across all time aggregations:")
print(avg_improvement.round(2))

# Find best performing aggregation for each model
print("\n" + "="*60)
print("Best performing time aggregation:")
print("="*60)
for metric in ["MAPE", "MAE", "RMSE"]:
    old_best = df_old[metric].idxmin()
    new_best = df_new[metric].idxmin()
    print(f"{metric}:")
    print(f"  NYISO-only: {old_best} ({df_old.loc[old_best, metric]:.4f})")
    print(f"  + Mesonet:  {new_best} ({df_new.loc[new_best, metric]:.4f})")

# R² - higher is better
old_best = df_old["R2"].idxmax()
new_best = df_new["R2"].idxmax()
print(f"R²:")
print(f"  NYISO-only: {old_best} ({df_old.loc[old_best, 'R2']:.4f})")
print(f"  + Mesonet:  {new_best} ({df_new.loc[new_best, 'R2']:.4f})")


In [ ]:
# Create detailed comparison table
comparison_table = pd.DataFrame()
for metric in metrics:
    comparison_table[f"{metric}_old"] = df_old[metric]
    comparison_table[f"{metric}_new"] = df_new[metric]
    if metric in ["MAPE", "MAE", "RMSE"]:
        comparison_table[f"{metric}_Δ%"] = improvement[f"{metric}_improvement_%"]
    else:
        comparison_table[f"{metric}_Δ%"] = improvement[f"{metric}_improvement_%"]

print("\nDetailed Comparison Table:")
print(comparison_table.round(4))

# Save to CSV
comparison_table.to_csv("model_comparison_detailed.csv")
print("\nSaved detailed comparison to: model_comparison_detailed.csv")
